# NLP Preprocessing


We will use a short excerpt from a corporate earnings call to demonstrate
common NLP preprocessing steps with spaCy. The excerpt is from Satya
Nadella's prepared remarks on Microsoft's earnings conference call for the
second quarter of fiscal year 2026, held on 28 January 2026. The text
follows the official transcript, which is linked from
[Microsoft's investor relations page for that call](https://www.microsoft.com/en-us/investor/events/fy-2026/earnings-fy-2026-q2).

In [1]:
# Excerpt from Satya Nadella's prepared remarks (Microsoft FY26 Q2 earnings call)
text = """
You can think of agents as the new Apps. And to build, deploy, and manage
agents, customers will need a model catalog, tuning services, harnesses for
orchestration, services for context engineering, AI safety, management,
observability, and security. It starts with having broad model choice. Our
customers expect to use multiple models as part of any workload that they
can fine-tune and optimize based on cost, latency, and performance
requirements. And we offer the broadest selection of models of any
hyperscaler. This quarter, we added support for GPT 5.2, as well as
Claude 4.5. Already, over 1,500 customers have used both Anthropic and
OpenAI models on Foundry. We are seeing increasing demand for
region-specific models, including Mistral and Cohere, as more customers
look for sovereign AI choices.
"""

:::{note} Why triple double quotes?

Using triple double quotes (`"""`) allows us to include multi-line text without needing to escape newlines or other special characters. This is especially useful for large blocks of text, such as transcripts, where we want to preserve the formatting and structure of the original content. You can also use triple single quotes (`'''`) for the same purpose.

:::


:::{note} Setup

This notebook uses spaCy and its small English model, `en_core_web_sm`. Both
are included in the course environment. If you have not set it up yet, run
this from the repository root:

```bash
python -m pip install -r requirements.txt
```

If you install spaCy on its own instead, download the model with
`python -m spacy download en_core_web_sm`.

:::

:::{note} What is `en_core_web_sm`?

`en_core_web_sm` is a small English language model provided by spaCy. It includes vocabulary, syntax, and entities, and is suitable for many NLP tasks. For more advanced features, larger models like `en_core_web_md` or `en_core_web_lg` can be used.

There are other language models available for different languages and sizes. You can find more information in the [spaCy documentation](https://spacy.io/models).

:::


## Common NLP Preprocessing Steps

Recall from the chapter overview that raw text often needs to be cleaned
before analysis. This notebook demonstrates these steps:

- Sentence segmentation
- Tokenization
- Lowercasing
- Removing stopwords, punctuation, and whitespace
- Lemmatization

[spaCy](https://spacy.io/) is a powerful NLP library that provides built-in
functions for many of these steps. It does not include a stemmer; it uses
lemmatization instead. Turning tokens into numbers (bag-of-words or TF-IDF
vectors) is shown in the **NLP in Business** notebook.

Load the model.


In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")

### Parse the Text as a spaCy Document

The first step is to parse the raw text into a spaCy `Doc` object.


In [3]:
doc = nlp(text)

### Sentence Segmentation

spaCy can automatically split text into sentences.


In [4]:
for sent in doc.sents:
    print("---------")
    print(sent.text.strip())

---------
You can think of agents as the new Apps.
---------
And to build, deploy, and manage
agents, customers will need a model catalog, tuning services, harnesses for
orchestration, services for context engineering, AI safety, management,
observability, and security.
---------
It starts with having broad model choice.
---------
Our
customers expect to use multiple models as part of any workload that they
can fine-tune and optimize based on cost, latency, and performance
requirements.
---------
And we offer the broadest selection of models of any
hyperscaler.
---------
This quarter, we added support for GPT 5.2, as well as
Claude 4.5.
---------
Already, over 1,500 customers have used both Anthropic and
OpenAI models on Foundry.
---------
We are seeing increasing demand for
region-specific models, including Mistral and Cohere, as more customers
look for sovereign AI choices.


### Tokenization

Tokenization splits text into individual units called tokens.


In [5]:
for token in doc[:10]:
    print(token.text)



You
can
think
of
agents
as
the
new
Apps


### Token Attributes

spaCy provides built-in metadata about each token.


In [6]:
for token in doc[:10]:
    print(token.text, token.pos_, token.is_stop)


 SPACE False
You PRON True
can AUX True
think VERB False
of ADP True
agents NOUN False
as ADP True
the DET True
new ADJ False
Apps NOUN False


We can format this into a Pandas DataFrame.


In [7]:
import pandas as pd

data = [
    {"token": token.text, "pos": token.pos_, "is_stop": token.is_stop}
    for token in doc[:10]
]

df = pd.DataFrame(data)
df

,token,pos,is_stop
0,\n,SPACE,False
1,You,PRON,True
2,can,AUX,True
3,think,VERB,False
4,of,ADP,True
5,agents,NOUN,False
6,as,ADP,True
7,the,DET,True
8,new,ADJ,False
9,Apps,NOUN,False


Useful attributes include:

| Attribute        | Description                    |
| ---------------- | ------------------------------ |
| `token.pos_`     | Part-of-speech                 |
| `token.lemma_`   | Base form of the word          |
| `token.is_stop`  | Whether the word is a stopword |
| `token.is_punct` | Whether it is punctuation      |
| `token.like_num` | Whether it represents a number |


### Lowercasing

spaCy tokens contain both original and normalized text.


In [8]:
tokens_lower = [token.text.lower() for token in doc if not token.is_punct]

tokens_lower[:10]

['\n', 'you', 'can', 'think', 'of', 'agents', 'as', 'the', 'new', 'apps']

### Removing Stopwords, Punctuation, and Whitespace

The `'\n'` tokens above come from the line breaks in our triple-quoted
string. spaCy keeps runs of whitespace other than a single space as tokens
of their own (with part of speech `SPACE`), and they are neither stopwords
nor punctuation. We drop them with `token.is_space`.

In [9]:
clean_tokens = [
    token.text.lower()
    for token in doc
    if not token.is_stop and not token.is_punct and not token.is_space
]

clean_tokens[:10]

['think',
 'agents',
 'new',
 'apps',
 'build',
 'deploy',
 'manage',
 'agents',
 'customers',
 'need']

### Lemmatization

Lemmatization reduces words to their base form.


In [10]:
lemmas = [
    token.lemma_.lower()
    for token in doc
    if not token.is_stop and not token.is_punct and not token.is_space
]

lemmas[:10]

['think',
 'agent',
 'new',
 'app',
 'build',
 'deploy',
 'manage',
 'agent',
 'customer',
 'need']

## Feature Extraction

spaCy can also extract more complex features like named entities, noun chunks, and syntactic dependencies.


### Part-of-Speech Tagging

spaCy can identify grammatical roles.


In [11]:
for token in doc[:10]:
    print(token.text, token.pos_)


 SPACE
You PRON
can AUX
think VERB
of ADP
agents NOUN
as ADP
the DET
new ADJ
Apps NOUN


### Named Entity Recognition

Named Entity Recognition (NER) identifies important entities in text.


In [12]:
for ent in doc.ents:
    print(ent.text, ent.label_)

Apps ORG
AI GPE
This quarter DATE
GPT ORG
Claude 4.5 PERSON
over 1,500 CARDINAL
OpenAI ORG
Foundry ORG
Mistral and Cohere ORG


:::{caution} Check entity output before you use it

Look closely at this output. The small model gets some entities right:
_This quarter_ (DATE), _over 1,500_ (CARDINAL), and _OpenAI_ (ORG). It also
makes mistakes that a human reader would not:

- _Apps_ is tagged as an organization, and _AI_ as a GPE (a country, city,
  or state).
- _Claude 4.5_ is tagged as a PERSON, probably because Claude is also a
  first name. It is an AI model, a product.
- _GPT_ is tagged as an organization, and its version number is dropped.
- _Mistral and Cohere_ becomes one organization instead of two.
- _Anthropic_ is not found at all.

`en_core_web_sm` is a small statistical model. Its entity recognizer was
trained on OntoNotes 5, a general English corpus released in 2013, long
before most of these AI companies and products existed. When entities matter
for an analysis, try a larger pipeline (`en_core_web_lg`, or the
transformer-based `en_core_web_trf`), add an `EntityRuler` with a list of
the company and product names you care about, or train on labeled examples
from your own documents. Either way, spot-check the output first.

:::

Entity types that appear above, plus a few other common ones (descriptions
from `spacy.explain`):

| Label    | Meaning                                       |
| -------- | --------------------------------------------- |
| PERSON   | People, including fictional                   |
| ORG      | Companies, agencies, institutions, etc.       |
| GPE      | Countries, cities, states                     |
| PRODUCT  | Objects, vehicles, foods, etc. (not services) |
| DATE     | Absolute or relative dates or periods         |
| CARDINAL | Numerals that do not fall under another type  |
| MONEY    | Monetary values, including unit               |

You can look up any label with `spacy.explain("ORG")`.

## Summary

In this notebook we took a short earnings-call excerpt through the usual
spaCy preprocessing steps:

- split it into sentences and tokens, and inspected token attributes;
- lowercased the tokens and removed stopwords, punctuation, and the
  whitespace tokens created by line breaks;
- reduced words to their lemmas (_agents_ became _agent_, _customers_ became
  _customer_);
- tagged parts of speech and extracted named entities, and saw that a small
  pretrained model makes clear entity mistakes on text about new companies
  and products.

Cleaned tokens and lemmas like these are the input to the techniques in the
**NLP in Business** notebook: lexicon scores, TF-IDF vectors, topic models,
and text classifiers.